# Demo: 2D image binary classification  

> SynapseMNIST3D dataset demo


In [ ]:
#| default_exp demo

In [1]:
from bioMONAI.data import *
from bioMONAI.transforms import *
from bioMONAI.core import *
from bioMONAI.core import Path
from bioMONAI.data import get_target, RandomSplitter
#from bioMONAI.nets import BasicUNet, DynUNet
from bioMONAI.losses import *
from bioMONAI.metrics import *
from bioMONAI.datasets import *
from bioMONAI.visualize import *

import medmnist
import os

#from monai.utils import set_determinism
from monai.transforms import ScaleIntensity

#set_determinism(0)

In [2]:
device = get_device()
print(device)

cuda


### Download and store the dataset

In [3]:
data_flag = 'synapsemnist3d'
data_path = Path('../_data/medmnist_data/')

# Ensure the save directory exists; create it if not
os.makedirs(data_path, exist_ok=True)

train_data, val_data, test_data = download_medmnist(data_flag,[],data_path)

# print training and validation datasets
print(train_data)
print("===================")
print(val_data)

Using downloaded and verified file: ../_data/medmnist_data/synapsemnist3d.npz
Using downloaded and verified file: ../_data/medmnist_data/synapsemnist3d.npz
Using downloaded and verified file: ../_data/medmnist_data/synapsemnist3d.npz
Dataset SynapseMNIST3D of size 28 (synapsemnist3d)
    Number of datapoints: 1230
    Root location: ../_data/medmnist_data
    Split: train
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'inhibitory synapse', '1': 'excitatory synapse'}
    Number of samples: {'train': 1230, 'val': 177, 'test': 352}
    Description: The SynapseMNIST3D is a new 3D volume dataset to classify whether a synapse is excitatory or inhibitory. It uses a 3D image volume of an adult rat acquired by a multi-beam scanning electron microscope. The original data is of the size 100×100×100um^3 and the resolution 8×8×30nm^3, where a (30um)^3 sub-volume was used in the MitoEM dataset with dense 3D mitochondria instance segmentation labels. Three neuroscience 

In [4]:
train_path = data_path/'train'
val_path = data_path/'val'

train_data.save(train_path)
val_data.save(val_path)

Saving train set to ../_data/medmnist_data/train/synapsemnist3d, csv_path=../_data/medmnist_data/train/synapsemnist3d.csv...


100%|██████████| 1230/1230 [00:32<00:00, 37.49it/s]


Saving val set to ../_data/medmnist_data/val/synapsemnist3d, csv_path=../_data/medmnist_data/val/synapsemnist3d.csv...


100%|██████████| 177/177 [00:04<00:00, 39.19it/s]


In [6]:
from fastai.vision.all import CategoryBlock, get_image_files, GrandparentSplitter, parent_label


batch_size = 128

data_ops = {
    'blocks':       (BioImageBlock(cls=BioImageBase), CategoryBlock),
    'get_items':    get_image_files,
    'splitter':     GrandparentSplitter(train_name=train_path/'synapsemnist3d', valid_name=val_path/'synapsemnist3d'),
    'get_y':        parent_label,
    'item_tfms':    [ScaleIntensity(), RandRot90(prob=0.5), RandFlip(prob=0.75)],
    'bs':           batch_size,
}

In [9]:
data = get_dataloader(
    train_path/'synapsemnist3d',
    show_summary=True,
    **data_ops,
)

Could not do one pass in your dataloader, there is something wrong in it. Please see the stack trace below:


IndexError: tuple index out of range

In [ ]:
train_x = BioImageStack(train_data.imgs)
print(train_x.shape)

train_y = train_data.labels
print(train_y.shape)

In [ ]:
# visualization
mosaic_image_3d(train_x[0],cmap='gray')

In [ ]:
from monai.data import DataLoader

train_loader = DataLoader([train_data, val_data], batch_size=128, shuffle=True)

In [ ]:
batch_size = 128

data_ops = {
    'blocks':       (BioImageBlock(cls=BioImageStack), CategoryBlock)
    #'get_items':    get_image_files,
    'splitter':     RandomSplitter(valid_pct=0.2),
    'item_tfms':    [ScaleIntensity(), RandRot90(prob=0.5), RandFlip(prob=0.75)],
    'bs':           batch_size,
}

data = get_dataloader(
    train_data, 
    show_summary=True,
    **data_ops,
    )

# print length of training and validation datasets
#print('train images:', len(data.train_ds.items), '\nvalidation images:', len(data.valid_ds.items))

In [ ]:
data.show_batch(max_n=2, cmap='hot')

### Load and train a 3D model

In [ ]:
import torch
from monai.networks.nets import DenseNet121
from torchmetrics.classification import BinaryAccuracy

net = DenseNet121(spatial_dims=3, in_channels=1, out_channels=1)

loss = torch.nn.CrossEntropyLoss()
metric = BinaryAccuracy()

trainer = fastTrainer(train_loader, net, loss_fn=loss, metrics=metric, show_summary=True)

In [ ]:
trainer.fit_flat_cos(500)

In [ ]:
trainer.show_results(cmap='gray')

In [ ]:
# trainer.save('tmp-model')

### Test data 
Evaluate the performance of the selected model on unseen data.
It’s important to not touch this data until you have fine tuned your model to get an unbiased evaluation!